# Milestone 6B: Structured Big Data Analytics with Spark SQL

This notebook explores anti-money laundering transaction networks using **Spark SQL** ANSI queries, window functions, and SQL temporary views.

### Key Queries:
1. **Fan-In Aggregators**: Sinks accumulating funds from numerous unique originators.
2. **Fan-Out Distributors**: Source accounts spraying funds across numerous accounts.
3. **Hourly Velocity Surges**: SQL Windowing with `LAG()` to detect rapid burstiness.
4. **Cross-Currency Arbitrage**: Inflow/outflow currency conversion discrepancies.
5. **Pass-Through Mule Signatures**: Dwell time < 2 hours with balanced volume.
6. **Composite Heuristic Scoring**: Multi-rule regulatory risk grading.


In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.spark_sql_analytics import create_spark_session, SparkSQLAnalytics

spark = create_spark_session('Databricks-M6B-SQL')

tx_path = '../data/processed/processed_df.csv'
features_path = '../data/processed/account_features.csv'

tx_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(tx_path)
features_df = spark.read.option('header', 'true').option('inferSchema', 'true').csv(features_path)

analytics = SparkSQLAnalytics(spark, tx_df, features_df)
print('Transactions view row count:', tx_df.count())
print('Features view row count:', features_df.count())


## 1. Fan-In Accumulation Hubs
Accounts receiving transfers from $\ge 5$ distinct senders.


In [ ]:
sql_df, dsl_df = analytics.query_fan_in_hubs(min_counterparties=5)
sql_df.show(10, truncate=False)


## 2. Windowed Velocity Bursts (SQL `LAG()` Window Functions)
Measures hour-over-hour transaction acceleration per account.


In [ ]:
velocity_df = analytics.query_hourly_velocity_bursts()
velocity_df.show(10, truncate=False)


## 3. Money Mule Pass-Through Signatures
Identifies accounts where inflow and outflow volumes are balanced within 25% and dwell time is under 2 hours.


In [ ]:
mules_df = analytics.query_pass_through_mules()
mules_df.show(10, truncate=False)


## 4. Rule-Based Composite Risk Scoring
Classifies accounts into CRITICAL, HIGH, MEDIUM, and LOW risk tiers using SQL `CASE WHEN` expressions.


In [ ]:
risk_df = analytics.query_composite_risk_scores()
risk_df.groupBy('rule_risk_tier').count().show()
risk_df.show(10, truncate=False)
spark.stop()
